In [7]:
import sys
sys.path.append('..')
 
import pandas as pd
import numpy as np
from scipy import stats
from src.data_loader import DataLoader
from src.statistics import StatisticalAnalyzer
 
loader = DataLoader(data_dir="../data")
df = pd.read_csv(loader.processed_dir / "cleaned_data.csv")
 
analyzer = StatisticalAnalyzer(df)
 
stats_df = analyzer.all_descriptive_stats()
stats_df

def comprehensive_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generate comprehensive statistics for all numeric columns.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    stats_dict = {
        'Column': [],
        'Mean': [],
        'Median': [],
        'Mode': [],
        'Std': [],
        'Variance': [],
        'Skewness': [],
        'Kurtosis': [],
        'Range': [],
        'IQR': [],
        'Q1': [],
        'Q3': [],
        'CV': [],
        'Missing %': []
    }
    
    for col in numeric_cols:
        data = df[col].dropna()
        
        stats_dict['Column'].append(col)
        stats_dict['Mean'].append(data.mean())
        stats_dict['Median'].append(data.median())
        stats_dict['Mode'].append(data.mode()[0] if not data.mode().empty else np.nan)
        stats_dict['Std'].append(data.std())
        stats_dict['Variance'].append(data.var())
        stats_dict['Skewness'].append(data.skew())
        stats_dict['Kurtosis'].append(data.kurtosis())
        stats_dict['Range'].append(data.max() - data.min())
        stats_dict['IQR'].append(data.quantile(0.75) - data.quantile(0.25))
        stats_dict['Q1'].append(data.quantile(0.25))
        stats_dict['Q3'].append(data.quantile(0.75))
        stats_dict['CV'].append(data.std() / data.mean() if data.mean() != 0 else np.nan)
        stats_dict['Missing %'].append(df[col].isnull().mean() * 100)
    
    return pd.DataFrame(stats_dict)

stats_df = comprehensive_stats(df)
stats_df.to_csv('../reports/summary_statistics.csv', index=False)
print("📊 Summary Statistics:")
print(stats_df.to_string())

def test_normality(df: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    Perform Shapiro-Wilk test for normality on numeric columns.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    results = {
        'Column': [],
        'Statistic': [],
        'P-value': [],
        'Normal?': [],
        'Interpretation': []
    }
    
    for col in numeric_cols:
        data = df[col].dropna()
        if len(data) > 3:
            stat, p_value = stats.shapiro(data)
            is_normal = p_value > alpha
            
            results['Column'].append(col)
            results['Statistic'].append(stat)
            results['P-value'].append(p_value)
            results['Normal?'].append(is_normal)
            results['Interpretation'].append(
                "Normal" if is_normal else "Not normal"
            )
    
    return pd.DataFrame(results)

normality_df = test_normality(df)
normality_df.to_csv('../reports/normality_tests.csv', index=False)
print("\n🔬 Normality Tests:")
print(normality_df.to_string())

def correlation_analysis(df: pd.DataFrame) -> dict:
    """
    Calculate correlation matrices using different methods.
    """
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    corr_matrix = df[numeric_cols].corr()
    
    spearman_matrix = df[numeric_cols].corr(method='spearman')
    
    corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            corr_value = corr_matrix.iloc[i, j]
            corr_pairs.append({
                'Variable 1': corr_matrix.columns[i],
                'Variable 2': corr_matrix.columns[j],
                'Correlation': corr_value,
                'Strength': abs(corr_value)
            })
    
    corr_pairs = sorted(corr_pairs, key=lambda x: x['Strength'], reverse=True)
    
    return {
        'pearson': corr_matrix,
        'spearman': spearman_matrix,
        'top_correlations': corr_pairs[:10]
    }

corr_results = correlation_analysis(df)

corr_results['pearson'].to_csv('reports/pearson_correlation.csv')
corr_results['spearman'].to_csv('reports/spearman_correlation.csv')

print("\n🔗 Top 5 Correlations:")
for i, pair in enumerate(corr_results['top_correlations'][:5], 1):
    print(f"{i}. {pair['Variable 1']} ↔ {pair['Variable 2']}: {pair['Correlation']:.3f}")

📊 Summary Statistics:
                  Column         Mean       Median     Mode         Std       Variance  Skewness  Kurtosis        Range          IQR           Q1          Q3        CV  Missing %
0       culmen_length_mm    43.925000    44.450000    41.10    5.443792      29.634869  0.051571 -0.863835    27.500000     9.225000    39.275000    48.50000  0.123934        0.0
1        culmen_depth_mm    17.152035    17.300000    17.00    1.969060       3.877197 -0.145196 -0.894380     8.400000     3.100000    15.600000    18.70000  0.114800        0.0
2      flipper_length_mm   200.892442   197.000000   190.00   14.023826     196.667698  0.351218 -0.971878    59.000000    23.000000   190.000000   213.00000  0.069808        0.0
3            body_mass_g  4200.872093  4050.000000  3800.00  799.696532  639514.543359  0.474832 -0.704620  3600.000000  1200.000000  3550.000000  4750.00000  0.190364        0.0
4        culmen_area_mm2   750.891453   731.245000   626.50  115.092974   13246.392

OSError: Cannot save file into a non-existent directory: 'reports'